## Лабораторная работа 2. Ассоциативные правила



In [1]:
import numpy as np
import pandas as pd
import time
from collections import defaultdict

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth as fpgrowth_mlxtend

DATA_PATH = "Online Retail.xlsx"

In [2]:
df_raw = pd.read_excel(DATA_PATH)
print(df_raw.head())
print("shape =", df_raw.shape)

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
shape = (541909, 8)


### Что такое «текущая потребительская корзина»

Под **текущей корзиной** будем понимать **набор товаров, которые покупатель уже положил в корзину в данный момент времени**.

Проверка рекомендаций:
- делим транзакции на `train` и `test` по чекам (InvoiceNo),
- на `train` строим частые наборы и правила,
- на `test` для каждой корзины берём один или несколько поднаборов товаров как "текущую корзину",
- по правилам генерируем рекомендации (товары в \(\text{Consequent}\), которых ещё нет в текущей корзине),
- проверяем, какие из рекомендованных товаров действительно присутствуют в полной корзине из `test` (это и есть «правильность» рекомендаций).

Так можно посчитать метрики качества рекомендаций (hit-rate, precision/recall и т.п.).

In [3]:
def preprocess_online_retail(df: pd.DataFrame) -> pd.DataFrame:
    """Очищаем и агрегируем данные до уровня корзин.
    Возвращает датафрейм с колонками:
    - InvoiceNo
    - items: список товаров в корзине
    """
    df = df.copy()

    df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
    df = df.dropna(subset=['InvoiceNo', 'StockCode', 'Quantity'])
    df = df[df['Quantity'] > 0]

    baskets = (
        df.groupby('InvoiceNo')['StockCode']
          .apply(lambda s: list(set(s.astype(str))))
          .reset_index(name='items')
    )

    baskets = baskets[baskets['items'].map(len) >= 2]
    baskets = baskets.reset_index(drop=True)
    return baskets


if df_raw is not None:
    baskets = preprocess_online_retail(df_raw)
    print("Число корзин:", len(baskets))
    baskets.head()

Число корзин: 18338


### Разбиение на train / test

Будем делить по корзинам (InvoiceNo), например, в пропорции 80/20 по времени или просто по индексу.

In [4]:
from typing import List, Tuple, Dict, FrozenSet, Optional


def train_test_split_baskets(baskets: pd.DataFrame, test_size: float = 0.2,
                             random_state: int | None = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(random_state)
    idx = np.arange(len(baskets))
    rng.shuffle(idx)
    split = int(len(idx) * (1 - test_size))
    train_idx = idx[:split]
    test_idx = idx[split:]
    return baskets.iloc[train_idx].reset_index(drop=True), baskets.iloc[test_idx].reset_index(drop=True)


if df_raw is not None:
    train_baskets, test_baskets = train_test_split_baskets(baskets, test_size=0.2)
    print("train:", len(train_baskets), "test:", len(test_baskets))

train: 14670 test: 3668


### Реализация Apriori

Реализуем классический алгоритм `Apriori` для поиска частых наборов:
- строим кандидаты \(C_k\) из частых наборов \(L_{k-1}\),
- считаем поддержку по транзакциям,
- фильтруем по порогу `min_support` (в долях от числа транзакций),
- получаем множество всех частых наборов и их поддержку.

In [5]:
def apriori(transactions: List[List[str]], min_support: float) -> Dict[FrozenSet[str], float]:
    """Поиск частых наборов товаров алгоритмом Apriori (ручная реализация).

    Возвращает словарь: itemset (frozenset) -> support (0..1).
    """
    n_transactions = len(transactions)
    min_count = int(np.ceil(min_support * n_transactions))

    transactions_sets = [set(t) for t in transactions]

    item_counts = defaultdict(int)
    for t in transactions_sets:
        for item in t:
            item_counts[frozenset([item])] += 1

    Lk = {itemset: cnt / n_transactions
          for itemset, cnt in item_counts.items()
          if cnt >= min_count}

    frequent_itemsets: Dict[FrozenSet[str], float] = dict(Lk)
    k = 2

    while Lk:
        Lk_itemsets = list(Lk.keys())
        Ck = set()
        for i in range(len(Lk_itemsets)):
            for j in range(i + 1, len(Lk_itemsets)):
                union = Lk_itemsets[i] | Lk_itemsets[j]
                if len(union) == k:
                    Ck.add(union)

        Ck_counts = defaultdict(int)
        for t in transactions_sets:
            for cand in Ck:
                if cand.issubset(t):
                    Ck_counts[cand] += 1

        Lk = {itemset: cnt / n_transactions
              for itemset, cnt in Ck_counts.items()
              if cnt >= min_count}

        frequent_itemsets.update(Lk)
        k += 1

    return frequent_itemsets


def build_transactions_from_baskets(df_baskets: pd.DataFrame) -> List[List[str]]:
    return df_baskets['items'].tolist()


if df_raw is not None:
    transactions_train = build_transactions_from_baskets(train_baskets)
    # One-hot для mlxtend FP-Growth
    te = TransactionEncoder()
    te_array = te.fit(transactions_train).transform(transactions_train)
    df_te = pd.DataFrame(te_array, columns=te.columns_)
    start = time.time()
    freq_itemsets_apriori = apriori(transactions_train, min_support=0.02)
    t_apriori = time.time() - start
    print(f"Apriori (ручной): найдено {len(freq_itemsets_apriori)} частых наборов, время {t_apriori:.2f} c")

Apriori (ручной): найдено 478 частых наборов, время 136.35 c


### FP-Growth (библиотека mlxtend)

Используем готовый алгоритм FP-Growth из `mlxtend.frequent_patterns`. Результат приводим к формату словаря для совместимости с `generate_rules`.

In [6]:
if df_raw is not None:
    start = time.time()
    freq_itemsets_fpgrowth_df = fpgrowth_mlxtend(df_te, min_support=0.02, use_colnames=True)
    t_fpgrowth = time.time() - start
    # Приводим к формату {frozenset -> support} для совместимости с generate_rules
    freq_itemsets_fpgrowth = dict(zip(freq_itemsets_fpgrowth_df['itemsets'], freq_itemsets_fpgrowth_df['support']))
    print(f"FP-Growth (mlxtend): найдено {len(freq_itemsets_fpgrowth)} частых наборов, время {t_fpgrowth:.2f} c")

FP-Growth (mlxtend): найдено 478 частых наборов, время 1.40 c


### Генерация ассоциативных правил

Имея частые наборы, построим правила вида \(A → B\), где \(A\) и \(B\) — непересекающиеся подмножества одного частого набора.

Для каждого правила считаем:
- поддержку (`support`),
- доверие (`confidence`),
- подъём (`lift`).

In [7]:
from itertools import combinations


def generate_rules(frequent_itemsets: Dict[FrozenSet[str], float],
                   min_confidence: float = 0.3):
    """Генерация ассоциативных правил из частых наборов (ручная).

    Возвращает список словарей с полями:
    - antecedent, consequent (frozenset)
    - support, confidence, lift
    """
    rules = []
    for itemset, supp in frequent_itemsets.items():
        # Приводим ключ к frozenset (на случай если пришёл str из другого формата)
        itemset = frozenset([itemset]) if isinstance(itemset, str) else frozenset(itemset)
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            for antecedent in combinations(itemset, r):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                if not consequent:
                    continue
                supp_A = frequent_itemsets.get(antecedent)
                supp_B = frequent_itemsets.get(consequent)
                if supp_A is None or supp_B is None or supp_A == 0:
                    continue
                confidence = supp / supp_A
                if confidence < min_confidence:
                    continue
                lift = confidence / supp_B
                rules.append({
                    'antecedent': antecedent,
                    'consequent': consequent,
                    'support': supp,
                    'confidence': confidence,
                    'lift': lift,
                })
    return rules


if df_raw is not None:
    rules_apriori = generate_rules(freq_itemsets_apriori, min_confidence=0.3)
    rules_fpgrowth = generate_rules(freq_itemsets_fpgrowth, min_confidence=0.3)
    print("Правил Apriori:", len(rules_apriori))
    print("Правил FP-Growth:", len(rules_fpgrowth))

Правил Apriori: 232
Правил FP-Growth: 232


### Функция рекомендаций по текущей корзине

Пусть текущая корзина \(C\) — это множество товаров, которые уже лежат в корзине.

Алгоритм рекомендаций:
- выбираем все правила \(A \Rightarrow B\) такие, что \(A \subseteq C\),
- в рекомендации добавляем товары из \(B\), которых ещё нет в \(C\),
- ранжируем по убыванию `confidence` (и, при желании, `lift`).

In [8]:
def recommend_from_rules(current_cart: List[str], rules, top_k: int = 10) -> List[Tuple[str, float, float]]:
    """Возвращает топ-k рекомендованных товаров для текущей корзины.

    rules: список словарей с полями antecedent, consequent, confidence, lift.
    """
    cart_set = set(map(str, current_cart))
    candidates: Dict[str, Tuple[float, float]] = {}

    for r in rules:
        A = set(map(str, r['antecedent']))
        B = set(map(str, r['consequent']))
        if A.issubset(cart_set):
            for item in B:
                if item in cart_set:
                    continue
                prev = candidates.get(item)
                if prev is None or r['confidence'] > prev[0]:
                    candidates[item] = (float(r['confidence']), float(r['lift']))

    recs = sorted(candidates.items(), key=lambda x: (-x[1][0], -x[1][1]))
    return [(item, conf, lift) for item, (conf, lift) in recs[:top_k]]


# Пример использования (после обучения правил):
if df_raw is not None and rules_apriori:
    example_cart = test_baskets.iloc[0]['items']
    print("Пример текущей корзины:", example_cart)
    recs = recommend_from_rules(example_cart, rules_apriori, top_k=5)
    print("Рекомендации (товар, confidence, lift):")
    for item, conf, lift in recs:
        print(item, f"conf={conf:.2f}", f"lift={lift:.2f}")

Пример текущей корзины: ['22998', '22804', '23243', '23240', '21877', '22994', '47593B', '85123A', '21733', '20725', '22916', '23236', '22413', '23005', '21165', '23300', '21868', '22921', '23004', '84596G', '22919', '22120', '21174', '21933', '35965', '23169', '21070', '85152', '22699', '23165', '21067', '22487', '20728', '22918', '23301', '21755', '22698', '21756', '22720', '23245', '21218', '22384', '21908', '23322', '23321', '22667', '22892', '23153', '84596B', '21875', '21745', '21871', '23014', '22383', '21068', '21166', '21932', '22423', '22920', '23167', '82583', '23152', '84596F', '21175', '79066K', '20726', '22917', '22907', '23013']
Рекомендации (товар, confidence, lift):
22697 conf=0.91 lift=16.39
20727 conf=0.55 lift=7.95
22382 conf=0.48 lift=7.53
85099B conf=0.38 lift=3.31
23206 conf=0.38 lift=6.53


### Оценка качества рекомендаций на test

Для проверки качества можно:
- для каждой корзины из `test` случайно выбрать подмножество товаров как текущую корзину `C`,
- сгенерировать рекомендации по правилам,
- сравнить рекомендованные товары с оставшимися товарами из исходной корзины,
- посчитать hit-rate / precision / recall.

Ниже — простая реализация hit-rate и средней точности.

In [9]:
from random import sample


def evaluate_recommendations(test_baskets: pd.DataFrame, rules, top_k: int = 5,
                             max_baskets: int | None = 500) -> dict:
    """Грубая оценка рекомендаций на test.

    Для каждой корзины берём случайное подмножество товаров как текущую корзину,
    оставшиеся товары считаем «истинно релевантными».
    """
    n = len(test_baskets)
    indices = list(range(n))
    if max_baskets is not None and max_baskets < n:
        indices = sample(indices, max_baskets)

    hits = 0
    total = 0
    total_precision = 0.0

    for idx in indices:
        items = list(map(str, test_baskets.iloc[idx]['items']))
        if len(items) < 2:
            continue
        # случайно делим на текущую корзину и будущие покупки
        cut = np.random.randint(1, len(items))
        current = items[:cut]
        future = set(items[cut:])
        if not future:
            continue

        recs = recommend_from_rules(current, rules, top_k=top_k)
        rec_items = [r[0] for r in recs]
        if not rec_items:
            continue

        rec_set = set(rec_items)
        hit = len(rec_set & future) > 0
        hits += int(hit)
        total += 1

        precision = len(rec_set & future) / len(rec_set)
        total_precision += precision

    if total == 0:
        return {"hit_rate": None, "mean_precision": None}

    return {
        "hit_rate": hits / total,
        "mean_precision": total_precision / total,
    }


if df_raw is not None and rules_apriori:
    metrics = evaluate_recommendations(test_baskets, rules_apriori, top_k=5)
    print("Hit-rate:", metrics["hit_rate"])
    print("Mean precision:", metrics["mean_precision"])

Hit-rate: 0.46835443037974683
Mean precision: 0.259563994374121


### Небольшие эксперименты с качеством рекомендаций
Меняем порог confidence, порог lift и top_k, не переобучая модели

In [10]:
from itertools import product


def filter_rules(rules_list, min_conf: float, min_lift: float):
    return [r for r in rules_list if r['confidence'] >= min_conf and r['lift'] >= min_lift]


if df_raw is not None and rules_apriori:
    results = []
    for min_conf, min_lift, top_k in product([0.3, 0.4, 0.5, 0.6], [1.0, 2.0, 3.0], [3, 5, 10]):
        rules_filtered = filter_rules(rules_apriori, min_conf=min_conf, min_lift=min_lift)
        if not rules_filtered:
            continue
        metrics = evaluate_recommendations(test_baskets, rules_filtered, top_k=top_k)
        results.append({
            "min_conf": min_conf,
            "min_lift": min_lift,
            "top_k": top_k,
            "n_rules": len(rules_filtered),
            "hit_rate": metrics["hit_rate"],
            "mean_precision": metrics["mean_precision"],
        })

    results_df = pd.DataFrame(results)
    display(results_df.sort_values(["hit_rate", "mean_precision"], ascending=False).head(20))

,min_conf,min_lift,top_k,n_rules,hit_rate,mean_precision
32,0.6,2.0,10,36,0.552381,0.495238
5,0.3,2.0,10,232,0.552124,0.241647
2,0.3,1.0,10,232,0.538776,0.212902
8,0.3,3.0,10,227,0.534483,0.244485
25,0.5,3.0,5,83,0.524390,0.380183
4,0.3,2.0,5,232,0.519531,0.249154
16,0.4,3.0,5,164,0.518987,0.276653
11,0.4,1.0,10,164,0.511211,0.264866
34,0.6,3.0,5,36,0.509259,0.427932
14,0.4,2.0,10,164,0.506276,0.268750
